# Projeto Retinopatia

Classificação binária de retinopatia diabética em fotos de fundo de olho,
usando transfer learning com EfficientNet.

Este notebook é apenas o **orquestrador da execução no Colab**: toda a lógica
vive nos módulos Python do repositório (`src/retinopatia/`), que são editados
no VS Code e trazidos para cá pelo `git clone`/`git pull` da primeira célula.

## Preparação do Repositório no Colab

In [ ]:
# O código do projeto vive no repositório do GitHub.
# Esta célula traz (ou atualiza) esse código dentro do Colab.

import os
import sys

URL_REPOSITORIO = "https://github.com/Rcorrea07/ML_Retinopatia.git"
PASTA_REPOSITORIO = "/content/ML_Retinopatia"

if not os.path.exists(PASTA_REPOSITORIO):
    !git clone {URL_REPOSITORIO} {PASTA_REPOSITORIO}
else:
    !git -C {PASTA_REPOSITORIO} pull

# Torna o pacote "retinopatia" importável neste notebook
caminho_src = os.path.join(PASTA_REPOSITORIO, "src")

if caminho_src not in sys.path:
    sys.path.insert(0, caminho_src)

# Recarrega os módulos automaticamente depois de um git pull,
# evitando ter que reiniciar o ambiente a cada edição no VS Code
%load_ext autoreload
%autoreload 2

print("Repositório pronto em:", PASTA_REPOSITORIO)

## Configurações Iniciais, Importações

In [ ]:
from retinopatia import (
    ambiente_colab,
    avaliacao,
    configuracao,
    conjunto_dados,
    pipeline_dados,
    pre_processamento,
    treinamento,
    utilitarios,
)

# Importada diretamente para não conflitar com a variável `modelo`
from retinopatia.modelo import construir_modelo

utilitarios.configurar_sementes(configuracao.SEED)

# Precisa vir antes da construção do modelo
utilitarios.configurar_precisao_mista()

## Conexão com o Drive e Preparação do Ambiente

In [ ]:
ambiente_colab.montar_drive()

ambiente_colab.extrair_dataset()

## Carregamento do CSV e Divisão Estratificada (Treino, Validação e Teste)

In [ ]:
dados, df_treino, df_val, df_teste = conjunto_dados.preparar_dataset()

## Inspeção visual

In [ ]:
pre_processamento.pre_visualizar_recorte(dados)

## Pré-processamento Offline (Evitar 40 min por época)

In [ ]:
resultados_pre_processamento = pre_processamento.pre_processar_dataset(dados)

## Data Generators (Pipelines de Dados e Data Augmentation)

In [ ]:
treino_data, val_data, teste_data = pipeline_dados.criar_geradores(
    df_treino,
    df_val,
    df_teste
)

pipeline_dados.verificar_escala(treino_data, val_data)

## Cálculo dos Pesos das Classes

In [ ]:
pesos_dicionario = treinamento.calcular_pesos_classe(df_treino)

## Arquitetura do Modelo (Transfer Learning com EfficientNet)

In [ ]:
modelo, modelo_base = construir_modelo()

## Callbacks (Monitorização do Treino)

In [ ]:
callbacks_fase1 = treinamento.criar_callbacks_fase1()

## Treino do Modelo (Fase 1 - Apenas as cabeças da rede)

In [ ]:
historico = treinamento.treinar_fase1(
    modelo,
    treino_data,
    val_data,
    pesos_dicionario,
    callbacks_fase1
)

## Avaliação, Gráficos e Matriz de Confusão (FASE 1)

In [ ]:
resultado_fase1 = avaliacao.avaliar_modelo(
    modelo,
    val_data,
    df_val,
    historico,
    "Fase 1"
)

## Fine-Tuning (Descongelando a Rede)

In [ ]:
callbacks_fase2 = treinamento.criar_callbacks_fase2()

historico_fase2 = treinamento.fine_tuning_fase2(
    modelo,
    modelo_base,
    treino_data,
    val_data,
    pesos_dicionario,
    callbacks_fase2
)

## Modelo completo

In [ ]:
# Salva o modelo completo: arquitetura + pesos
modelo.save(configuracao.CAMINHO_MODELO_FINAL)

print("Modelo completo salvo com sucesso!")

ambiente_colab.baixar_arquivo(configuracao.CAMINHO_MODELO_FINAL)

## Avaliação, Gráficos e Matriz de Confusão (FASE 2)

In [ ]:
resultado_fase2 = avaliacao.avaliar_modelo(
    modelo,
    val_data,
    df_val,
    historico_fase2,
    "Fase 2"
)

In [ ]:
# Estimativa final: limiares escolhidos na validação, aplicados ao teste
resultado_teste = avaliacao.avaliar_no_teste(
    modelo,
    teste_data,
    df_teste,
    resultado_fase2
)